In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shayanshahid997/yellow-taxi-trip-record-of-january-2024")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\kamek\.cache\kagglehub\datasets\shayanshahid997\yellow-taxi-trip-record-of-january-2024\versions\1


In [2]:
import os
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pyspark.sql import SparkSession

spark_master_address = "spark://localhost:7077"
spark = (
    SparkSession
    .builder
    .appName("NYCTaxiDataProcessing")
    .master("local[*]")  # .master(spark_master_address)
    .config("spark.hadoop.hadoop.native.io", "false")
    .getOrCreate()
)

In [3]:
parquet_path = os.path.join(path, "yellow_tripdata_2024-01.parquet")

In [4]:
taxi_df = spark.read.parquet(parquet_path)

In [5]:
taxi_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

# Operation showcase

### Time-aware dynamic windows

In [6]:
from pyspark.sql.functions import window, avg

result_df = (
    taxi_df
    .withColumn("tpep_pickup_datetime", taxi_df["tpep_pickup_datetime"].cast("timestamp"))
    .groupBy(
        window("tpep_pickup_datetime", "1 hour")
    )
    .agg(
        avg("fare_amount").alias("avg_fare")
    )
    .orderBy("window")
)

result_df.show()


+--------------------+------------------+
|              window|          avg_fare|
+--------------------+------------------+
|{2002-12-31 22:00...|               0.0|
|{2009-01-01 00:00...|              50.6|
|{2009-01-01 23:00...|              24.7|
|{2023-12-31 23:00...|             14.41|
|{2024-01-01 00:00...|18.977327167980604|
|{2024-01-01 01:00...|20.460195785180165|
|{2024-01-01 02:00...| 19.70491639871386|
|{2024-01-01 03:00...|20.030449756888192|
|{2024-01-01 04:00...|20.456678168130527|
|{2024-01-01 05:00...|21.630328515111668|
|{2024-01-01 06:00...| 26.31200391644908|
|{2024-01-01 07:00...| 27.19546444121913|
|{2024-01-01 08:00...|27.235174418604647|
|{2024-01-01 09:00...| 24.90943356643357|
|{2024-01-01 10:00...|22.612615996464875|
|{2024-01-01 11:00...| 20.02552118359111|
|{2024-01-01 12:00...|18.920660804689582|
|{2024-01-01 13:00...| 21.13468073737644|
|{2024-01-01 14:00...| 21.95912529002319|
|{2024-01-01 15:00...|22.420105376344097|
+--------------------+------------

### AVG trip distance per vendor

In [7]:
(
    taxi_df
    .select(['VendorID', 'trip_distance'])
    .groupBy('VendorID')
    .agg(avg('trip_distance').alias('avg_trip_distance'))
    .sort('avg_trip_distance')
    .show()
)

+--------+------------------+
|VendorID| avg_trip_distance|
+--------+------------------+
|       1|2.9747373008172904|
|       2|3.8724931979852943|
|       6|11.346846153846151|
+--------+------------------+



### ABG tip distance per vendor - SQL
In pyspark we can use SQL syntax to perform operations as well.

In [8]:
taxi_df.createOrReplaceTempView("taxi_df_view")

(
    spark
    .sql(
    """
    select VendorID, avg(trip_distance) as avg_trip_distance
    from taxi_df_view
    group by VendorID
    order by avg_trip_distance
    """
    )
    .show()
)

+--------+------------------+
|VendorID| avg_trip_distance|
+--------+------------------+
|       1|2.9747373008172904|
|       2|3.8724931979852943|
|       6|11.346846153846151|
+--------+------------------+



# Multi table operations
## Show correlation between demand and temperature
Show difference of taxi demand with correlation to temperature below month average, and precipitation above month average.

#### Load DFs

Taxi data

In [9]:
import kagglehub
# Download all 2024 taxi data data

path = kagglehub.dataset_download("sygnation/nyc-yellow-taxi-records-2024")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\kamek\.cache\kagglehub\datasets\sygnation\nyc-yellow-taxi-records-2024\versions\1


In [10]:
import os

# Due to spark limitation on Windows, I do load files with paths, not with directory path
files = [os.path.join(path, x) for x in os.listdir(path) if x.endswith('.parquet')]
taxi_df = spark.read.parquet(*files)

In [11]:
# taxi_df.show(5)

Weather data

In [12]:
# Loading latest weather data for 2024 in New York City
weather_df = spark.read.csv("data/nyc-2024-weather.csv", header=True, inferSchema=True)
# weather_df.show(5)

#### Prepare dataframes

In [13]:
from pyspark.sql import functions as F

taxi_df_per_day = (
    taxi_df
    .filter(
        F.year("tpep_pickup_datetime") == 2024
    )
    .sort('tpep_pickup_datetime')
    .groupBy(F.window("tpep_pickup_datetime", "1 day"))
    .agg(
        F.count("*").alias("total_trips"),
        F.sum('trip_distance').alias("daily_distance"),
        F.avg('trip_distance').alias("avg_daily_distance"),
        F.sum('fare_amount').alias("daily_fare_amount"),
        F.avg('fare_amount').alias("avg_daily_fare_amount"),
    )
    .withColumn("date", F.col("window.start").cast("date"))
    .sort("date")
    .drop("window")
)


# taxi_df_per_day.show(5)

In [14]:
from pyspark.sql import functions as F

weather_df_ny = (
    weather_df
    .withColumn(
        "DATE", F.to_timestamp("DATE", format="yyyy-MM-dd")
    )
    .select([
        F.col('DATE').alias("date"),
        F.round(((((F.col("TMAX") + F.col("TMIN")) / 2) - 32) * 5/9), 2).alias("avg_temp"),  # Calc mean and convert to C
        F.round((F.col('PRCP') * 25.4), 2).alias('precipitation')  # Convert inch to mm
    ])
)

# weather_df_ny.show(5)

#### Calculations
Transform DF to present avg daily distance per weekday

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# I decided to go with in built function to get weekday name, but you can use custom function as well
@F.udf(returnType=StringType())
def cast_weekday_to_week_name(x):
    return weekday_names[x]

combined_df = (
    taxi_df_per_day
    .join(
        weather_df_ny,
        on='date',
        how='left'
    )
    .withColumn("weekday_num", ((F.dayofweek("date") + 5) % 7) + 1)
    .withColumn("weekday", F.date_format("date", "EEEE"))
    .sort('weekday_num')
)

In [16]:
from pyspark.sql import functions as F

avg_week_distance = (
    combined_df
    .groupBy('weekday', "weekday_num")
    .agg(
        F.avg('avg_daily_distance').alias("avg_daily_distance"),
    )
    .withColumn(
        "source", F.lit("Average")
    )
    .sort("weekday_num")
)

avg_week_distance_lower_temps = (
    combined_df
    # Filter temps than lower avg
    .join(
        combined_df.select(F.avg('avg_temp').alias('year_avg_temp')), how='cross'
    )
    .filter(
        F.col('avg_temp') < F.col('year_avg_temp')
    )
    .groupBy('weekday', 'weekday_num')
    .agg(
        F.avg('avg_daily_distance').alias("avg_daily_distance"),
    )
    .withColumns({
        "source": F.lit("Lower temps")
    })
    .sort("weekday_num")
)

avg_week_distance_higher_precipitation = (
    combined_df
    .join(
        combined_df.select(F.avg('precipitation').alias('year_precipitation')), how='cross'
    )
    .filter(
        F.col('precipitation') < F.col('year_precipitation')
    )
    .groupBy('weekday', 'weekday_num')
    .agg(
        F.avg('avg_daily_distance').alias("avg_daily_distance"),
    )
    .withColumn(
        "source", F.lit("Higher precipitation")
    )
    .sort("weekday_num")
)


In [17]:
avg_week_distance.show(10)

+---------+-----------+------------------+-------+
|  weekday|weekday_num|avg_daily_distance| source|
+---------+-----------+------------------+-------+
|   Monday|          1|  5.32873884708305|Average|
|  Tuesday|          2| 4.758713579848455|Average|
|Wednesday|          3| 4.590561214393135|Average|
| Thursday|          4|  4.67810438321124|Average|
|   Friday|          5| 5.080648860993088|Average|
| Saturday|          6| 4.783188808791744|Average|
|   Sunday|          7| 5.628135706389319|Average|
+---------+-----------+------------------+-------+



### Results

In [18]:
# Collect df
spark_df = (
    avg_week_distance
    .union(avg_week_distance_lower_temps)
    .union(avg_week_distance_higher_precipitation)
)

pandas_df = spark_df.toPandas()

In [19]:
pandas_df.head(90)

,weekday,weekday_num,avg_daily_distance,source
0,Monday,1,5.328739,Average
1,Tuesday,2,4.758714,Average
2,Wednesday,3,4.590561,Average
3,Thursday,4,4.678104,Average
4,Friday,5,5.080649,Average
5,Saturday,6,4.783189,Average
6,Sunday,7,5.628136,Average
7,Monday,1,5.049228,Lower temps
8,Tuesday,2,4.486777,Lower temps
9,Wednesday,3,4.416743,Lower temps


In [20]:
import plotly.express as px

chart = (
    px.line(
        pandas_df,
        x="weekday",
        y="avg_daily_distance",
        color="source",
        markers=True,
        title="Average Daily Distance per Weekday by Weather Condition"
    )
)

chart.update_layout(
    title={
        'text': "Average Daily Distance per Weekday by Weather Condition",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=22)
    },
    xaxis_title="Weekday",
    yaxis_title="Average Distance (miles)",
    legend_title_text="Condition",
    font=dict(
        family="Arial",
        size=14
    ),
)

chart.write_image("chart.png")

chart.show()


![Plotly Chart](chart.png)


### Analysis

The graph shows Average Daily Distance per Weekday by Weather Condition:

- Average weather and higher precipitation conditions result in similar distances throughout the week, peaking on Sunday at over 5.6 miles.
- Lower temperature days consistently show reduced activity, especially from Tuesday to Saturday, with the lowest point on Saturday (~4.45 miles).
- Across all conditions, Sunday tends to have the highest distance, suggesting increased activity on weekends regardless of weather.
- Lower temperatures appear to have a greater negative impact on daily distance than higher precipitation.